# Analysis of samples from TNP-KR

## TODO

$$
\def\ctx{\mathrm{ctx}}
\def\test{\mathrm{test}}
\def\autoreg{\mathrm{autoreg}}
$$

- plot difference heatmaps
- plot monte carlo traces (do things even converge?)
- forward error? i.e. distribution of $q_\theta^\autoreg(\test | \ctx)p(\ctx)$ vs $p(\ctx, \test)$ vs $q_\theta(\test | \ctx)p(\ctx)$


## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from omegaconf import OmegaConf
from scipy import stats
from sps.kernels import rbf

results_dir = Path(
    "/Users/pgrynfelder/Library/CloudStorage/GoogleDrive-wadh6460@ox.ac.uk/My Drive/results"
)


runs = sorted(
    list(set(map(lambda x: x.parent, results_dir.glob("*/*.npy")))),
    key=lambda path: path.stat().st_mtime,
)

In [ ]:
# load model

prompt = "Which run? Type a number.\n" + "\n".join(
    [f"{i}: {run.name}" for i, run in enumerate(runs)]
)
print(prompt)
i = int(input(prompt))
run = runs[i]
print(f"Selected {run.name}")

# Load data
data = dict()
for file in run.glob("*.npy"):
    data[file.stem] = np.load(file)

# we know by definition what the diagonal samples are - unless debugging, we don't need them
data.pop("diagonal_paths", None)
data.pop("diagonal_densities", None)

# Simplify dimensionality - this can break if we have e.g. only one location
for key in data.keys():
    data[key] = data[key].squeeze()


# Sort s_test for easier plotting
idx = np.argsort(data["s_test"])
data["s_test"] = data["s_test"][idx]
for key in data.keys():
    if key.endswith("_paths"):
        data[key] = data[key][:, idx]
data["diagonal_mu"] = data["diagonal_mu"][idx]
data["diagonal_var"] = data["diagonal_var"][idx]


print("Seed:", data["seed"])
for key, value in sorted(data.items()):
    print(key, value.shape)

if (run / "config.yaml").exists():
    config = OmegaConf.load(run / "config.yaml")
else:
    config = None

paths = sorted([key for key in data.keys() if key.endswith("_paths")])

summary = pd.DataFrame()

In [ ]:
# Print config
from json import dumps

print(dumps(OmegaConf.to_object(config), indent=4))
print("Variance", data["var"])
print("Lengthscale", data["ls"])


In [ ]:
# Utils
from jax.experimental import enable_x64


def grid(num_items: int, ncols: int = 3, **kwargs):
    """`plt.subplots` with better defaults"""
    nrows = (num_items - 1) // ncols + 1
    figsize = (ncols * 5, nrows * 5)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kwargs)
    fig.set_constrained_layout(True)
    for ax in axes.flat[num_items:]:
        ax.axis("off")
    return fig, axes


def logpdf(s, f, kernel, var, ls):
    s = s.astype(np.float64)
    f = f.astype(np.float64)
    with enable_x64():
        cov = kernel(s, s, var, ls)

    return stats.multivariate_normal.logpdf(f, cov=cov)

## Analytic solution

This explicitly computes
$ p( f_{1:n} \mid \mathrm{ctx})$

Note that it assumes we know the kernel, variance, lengthscale, and observation noise so requires strictly more information than the model.

In [ ]:
from dl4bi.meta_learning.autoregressive.analytic import analytic_gp

analytic_mu, analytic_cov = analytic_gp(
    data["s_ctx"],
    data["f_ctx"],
    data["s_test"],
    kernel=rbf,
    var=data["var"],
    ls=data["ls"],
    obs_noise=config.data.obs_noise,
)
analytic_mu, analytic_cov = np.array(analytic_mu), np.array(analytic_cov)

data["analytic_mu"] = analytic_mu
data["analytic_cov"] = analytic_cov

print(
    f"Analytic solution's rank: {np.linalg.matrix_rank(analytic_cov, hermitian=True)} of {analytic_cov.shape[0]}"
)

## Metrics

See notes on why nothing really but average log-likelihood + some qualitative results makes sense.

## average log-likelihood of data
TODO

This is the metric that makes sense, see notes.

## Covariance

In [ ]:
from matplotlib.colors import SymLogNorm

logscale = 0.0  # if not None, symmetric log scale at this value away will be used
draw_context = True

covariances = {
    "diagonal": np.diag(data["diagonal_var"]),
}
for key in paths:
    covariances[key] = np.cov(data[key].T)

vmin = min([image.min() for image in covariances.values()])
vmax = max([image.max() for image in covariances.values()])
v = max(abs(vmin), abs(vmax))
cmap = plt.get_cmap("RdBu")
if logscale:
    norm = SymLogNorm(vmin=-v, vmax=v, linthresh=logscale)
else:
    norm = plt.Normalize(vmin=-v, vmax=v)
im = plt.cm.ScalarMappable(norm=norm, cmap=cmap)


# draw context points between test points
ctx_coordinates = []
if draw_context:
    for s in data["s_ctx"]:
        i = np.searchsorted(data["s_test"], s)
        if i == 0 or i == len(data["s_test"]):
            ctx_coordinates.append(i)
        lo, hi = data["s_test"][i - 1], data["s_test"][i]
        ctx_coordinates.append(i + (hi - s) / (hi - lo))

fig, axes = grid(len(covariances) * 3, ncols=3, sharex=True, sharey=True)
for (path, image), row in zip(covariances.items(), axes):
    rank = np.linalg.matrix_rank(image, hermitian=True)
    frobenius = np.linalg.norm(image - data["analytic_cov"], ord="fro")
    summary.loc[path, "rank"] = rank
    summary.loc[path, "frobenius"] = frobenius

    row[0].set_title("analytic")
    row[0].set_xlabel(
        f"rank = {np.linalg.matrix_rank(data['analytic_cov'], hermitian=True)}"
    )
    row[0].imshow(data["analytic_cov"], cmap=cmap, norm=norm)
    row[1].set_title(path)
    row[1].set_xlabel(f"rank = {rank}")
    row[1].imshow(image, cmap=cmap, norm=norm)
    row[2].set_title("difference")
    row[2].set_xlabel(f"$\| \cdot \|_F$ = {frobenius:.4f}")
    row[2].imshow(image - data["analytic_cov"], cmap=cmap, norm=norm)

    for ax in row:
        ax.hlines(ctx_coordinates, *ax.get_xlim(), "g", linewidth=0.2)
        ax.vlines(ctx_coordinates, *ax.get_ylim(), "g", linewidth=0.2)
        # ax.axis("off")

    fig.colorbar(im, ax=row)

fig.suptitle(
    "Covariance matrices for different path sampling strategies + analytic solutions"
)
print(summary[["frobenius"]].sort_values("frobenius", ascending=True))
plt.show()


## Means + 95% CI

In [ ]:
means = {
    "analytic": data["analytic_mu"],
    "diagonal": data["diagonal_mu"],
} | {path: np.mean(data[path], axis=0) for path in paths}
variances = {
    "analytic": np.diag(data["analytic_cov"]),
    "diagonal": data["diagonal_var"],
} | {path: np.var(data[path], axis=0) for path in paths}


fig, axes = grid(len(means))

for path, ax in zip(means.keys(), axes.flat):
    mu = means[path]
    var = variances[path]
    lo = mu - 2 * np.sqrt(var)
    hi = mu + 2 * np.sqrt(var)
    MSE = ((mu - data["analytic_mu"]) ** 2).mean()
    ax.plot(data["s_test"], mu, alpha=0.5)
    ax.fill_between(data["s_test"], lo, hi, alpha=0.2)
    ax.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context", linewidth=500)
    ax.plot(data["s_test"], data["analytic_mu"], "--", label="analytic mean", alpha=0.5)
    ax.legend()
    ax.set_title(f"{path}, MSE={MSE:.4f}")

fig.suptitle("$\mu \pm 2\sigma$")
plt.show()


## Mean

In [ ]:
fig, axes = grid(len(means))
for (path, mean), ax in zip(means.items(), axes.flat):
    ax.set_title(path)
    ax.plot(data["s_test"], mean)
    ax.plot(data["s_test"], data["analytic_mu"], "--", label="Analytic", alpha=0.5)
    ax.plot(data["s_ctx"], data["f_ctx"], "g+", label="Context")

fig.suptitle("$\mu$")
plt.legend()
plt.plot()

## Variance

In [ ]:
fig, axes = grid(len(variances))
for (path, variance), ax in zip(variances.items(), axes.flat):
    ax.set_title(path)
    ax.plot(data["s_test"], variance)
    # ax.plot(
    #     data["s_test"],
    #     np.diag(data["analytic_cov"]),
    #     "--",
    #     alpha=0.5,
    #     label="Analytic variance",
    # )
    ax.plot(data["s_ctx"], 0 * data["s_ctx"], "g+", label="Context")
fig.set_constrained_layout(True)
fig.suptitle("Variance")
plt.legend()
plt.show()

## Example sample paths

In [ ]:
Np = 5

indices = np.random.choice(data["ltr_paths"].shape[0], (Np,), replace=False)

fig, axes = grid(len(paths) * Np, ncols=Np, sharey=True, sharex=True)
for path, axs in zip(paths, axes):
    axs[0].set_ylabel(path)
    for idx, ax in zip(indices, axs):
        ax.plot(data["s_test"], data[path][idx])
        ax.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context")
fig.suptitle("Autoregressive sample paths")

plt.show()

In [ ]:
plt.title("A path from the true gp")

idx = data["s"].argsort()
plt.plot(data["s"][idx], data["f"][idx], "-")
plt.plot(data["s_ctx"], data["f_ctx"], "g.", label="Context")

## Normality testing

### Shapiro-Wilk test for normality

This is expected to fail almost always for large sample sizes.

In [ ]:
# %pip install mvshapirotest mvem
from mvShapiroTest.test import mvshapiro

for path in paths:
    num_paths = 4500  # need to truncate to avoid numerical instability
    statistic, pvalue = stats.shapiro(data[path][:4000], axis=0)
    alpha = 0.05
    summary.loc[path, "SW avg statistic"] = statistic.mean()
    summary.loc[path, "SW avg p-value"] = pvalue.mean()
    summary.loc[path, "SW non-normal"] = (pvalue < alpha).sum()
    summary.loc[path, "SW alpha"] = alpha
    result = mvshapiro(data[path][:num_paths] / 100)

    summary.loc[path, "mvS statistic"] = result["statistic"]
    summary.loc[path, "mvS p-value"] = result["p_value"]

summary[
    [
        "SW avg statistic",
        "SW avg p-value",
        "SW non-normal",
        "SW alpha",
        "mvS statistic",
        "mvS p-value",
    ]
]

### plot qq plots

In [ ]:
N_locations = 100
locations = sorted(
    np.random.choice(data["s_test"].shape[0], N_locations, replace=False)
)
path = "paths_random"

fig, axes = grid(N_locations, ncols=10)
for location, ax in zip(locations, axes.flat):
    _, (slope, intercept, _) = stats.probplot(
        data[path][:, location], dist="norm", plot=ax
    )
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"QQ plots of value of {path} at {N_locations} randomly chosen locations")
print("Context:", np.sort(data["s_ctx"]))
plt.plot()

### plot histograms

In [ ]:
fig, axes = grid(N_locations, ncols=5)
for location, ax in zip(locations, axes.flat):
    ax.hist(data[path][:, location], bins="auto", density=True)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")
fig.suptitle(
    f"Histograms of value of {path} at {N_locations} randomly chosen locations"
)
plt.plot()

## Estimation traces (to see whether estimates converge)

In [ ]:
path = "paths_random"
N_locations = 100
ns = np.arange(11, data[path].shape[0] + 1)  # dropping first 10 steps from the plot

np.random.seed(0)
locations = np.random.choice(data["s_test"].shape[0], N_locations, replace=False)
locations.sort()

### running means

In [ ]:
fig, axes = grid(N_locations, ncols=5)
for location, ax in zip(locations, axes.flat):
    running_sum = data[path][:, location].cumsum()[ns - 1]
    running_mean = running_sum / ns
    ax.plot(ns, running_mean, label="Running mean", linewidth=2)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"Running means for {path} at {N_locations} randomly chosen locations")
plt.show()


### plot running variances

In [ ]:
ns = np.arange(
    11, data[path].shape[0] + 1
)  # note dropping the first 10 steps from the plots
running_variances = np.stack([np.var(data[path][:n, locations], axis=0) for n in ns]).T

fig, axes = grid(N_locations, ncols=5)
for location, ax, running_variance in zip(locations, axes.flat, running_variances):
    ax.plot(ns, running_variance)
    ax.set_title(f"$s = {data['s_test'][location]:.2f}$")

fig.suptitle(f"Running variances for {path}")
plt.show()